# Lab 1 exercise solution - AI tutor with OpenAI Agents SDK

This is an OpenAI Agents SDK version of my Week 1 simulated AI tutor for absolute beginners.

The tutor answers questions from my course notes, politely declines topics that are not covered, and records an email address when someone wants to get in touch. The behavior stays the same, but the Agents SDK now provides the agent loop and tool handling.

In [ ]:
from pathlib import Path

from dotenv import load_dotenv
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr

from agents import Agent, Runner, trace, function_tool

load_dotenv(override=True)

In [ ]:
pdf_name = "agentic_ai_foundations_theory_mcp.pdf"
local_pdf = Path(pdf_name)
repo_pdf = Path("2_openai/community_contributions/Andras_Nemes") / pdf_name
pdf_path = local_pdf if local_pdf.exists() else repo_pdf

if not pdf_path.exists():
    raise FileNotFoundError(f"Could not find {pdf_name}")

reader = PdfReader(pdf_path)
notes = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        notes += text + "\n"

print(f"Loaded {len(reader.pages)} pages and {len(notes)} characters of notes")

In [ ]:
tutor_instructions = f"""
# Your role

You are a friendly and patient tutor for learners of agentic AI, running on a website.
Your visitors are absolute beginners who want to understand the basics: what agents are,
what tools are, how LLMs fit into the picture, what MCP is, and so on.

# Your style

Assume no prior knowledge of agentic AI. Explain concepts in simple, everyday language.
Introduce technical terms gently and explain them when you first use them.
Keep answers short and clear, and prefer a concrete example over an abstract definition.
Be warm and encouraging.

# Your material

Base your answers on these course notes:

{notes}

# Rules

Only answer based on the course notes above. If a question cannot be answered from the notes,
politely say that the topic is not covered in your material. Never make up an answer.
Suggest a related topic from the notes that the visitor could ask about instead.

If a question is not about agentic AI, politely steer the conversation back to agentic AI.

If a visitor would like to get in touch to chat about agentic AI, ask for their email address.
When they provide it, use the record_email tool and then confirm that it has been recorded.
"""

## The email registration tool

In [ ]:
emails_path = pdf_path.with_name("emails.txt")

@function_tool
def record_email(email: str) -> str:
    """Record the email address of a visitor who wants to discuss agentic AI.

    Args:
        email: The visitor's email address.
    """
    print(f"Tool called to record an email: {email}")
    with emails_path.open("a", encoding="utf-8") as file:
        file.write(email + "\n")
    return "Email received"

In [ ]:
print(record_email.name)
print(record_email.description)

## Make the tutor agent

In [ ]:
tutor = Agent(
    name="Beginner AI Tutor",
    instructions=tutor_instructions,
    model="gpt-5.4-mini",
    tools=[record_email],
)

## Test 1: a beginner question covered by the notes

In [ ]:
with trace("AI tutor - covered question"):
    result = await Runner.run(tutor, "I'm completely new to this. What is an AI agent?")

display(Markdown(result.final_output))

## Test 2: a question not covered by the notes

In [ ]:
with trace("AI tutor - unsupported question"):
    result = await Runner.run(tutor, "How do I fine-tune my own LLM from scratch?")

display(Markdown(result.final_output))

## Test 3: calling the email tool

In [ ]:
with trace("AI tutor - email registration"):
    result = await Runner.run(
        tutor,
        "This was helpful. I'd like to chat more about agentic AI. My email is test@example.com",
    )

display(Markdown(result.final_output))

## Add conversation memory to the Gradio chat

In [ ]:
async def chat(message, history):
    next_input = history + [{"role": "user", "content": message}]

    with trace("AI tutor chat"):
        result = await Runner.run(tutor, next_input)

    return result.final_output

## Launch the tutor

Try a few follow-up questions to confirm that the tutor remembers the conversation. You can inspect each turn at https://platform.openai.com/traces.

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)